In [0]:
# # Cell 1: Imports
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# from matplotlib.patches import Rectangle

# # Cell 2: Create Widget
# dbutils.widgets.text("country", "Germany", "Country")

# # Cell 3: Get Country Parameter
# country = dbutils.widgets.get("country")
# print(f"Selected Country: {country}")

# # Cell 4: Load Supply Data
# supply_query = f"""
# SELECT *
# FROM production.supply_analytics.dst_supply_dataset
# WHERE country_name = '{country}'
# """

# supply_df = spark.sql(supply_query).toPandas()
# print(f"Loaded {len(supply_df)} supply rows for {country}")

# # Cell 5: Load Price Parity Data
# parity_query = f"""
# SELECT *
# FROM production.supply_analytics.dst_price_parity
# WHERE country_name = '{country}'
# """

# parity_df = spark.sql(parity_query).toPandas()
# print(f"Loaded {len(parity_df)} parity rows for {country}")

# print("\nSupply data:")
# print(supply_df)
# print("\nParity data:")
# print(parity_df)

# # Cell 6: Plot Combined DID Chart
# def plot_combined_did_chart(
#     supply_df,
#     parity_df,
#     country,
#     period_col="period_window",
#     treat_period="CY",
#     control_period="LY",
#     pre_window="PRE",
#     post_window="POST",
#     figsize=(10, 7),  # REDUCED from (10, 12)
#     strip_left=0.75,
#     strip_width=0.25,
#     strip_fontsize=8,  # REDUCED from 9
# ):
#     """
#     Combined DID chart: Supply + Price Parity metrics in one chart
#     """
    
#     # Define all metrics and their groupings
#     metrics_config = [
#         # Supply metrics (Extensive Margin)
#         ("total_tours", "Total tours", "Extensive Margin", "supply"),
#         ("active_tours", "Active tours", "Extensive Margin", "supply"),
#         ("share_active_tours", "Share active tours", "Extensive Margin", "supply"),
#         ("total_suppliers", "Total suppliers", "Extensive Margin", "supply"),
#         ("active_suppliers", "Active suppliers", "Extensive Margin", "supply"),
#         ("share_active_suppliers", "Share active suppliers", "Extensive Margin", "supply"),
        
#         # Price parity metrics (Customer Response)
#         ("supplier_parity_rate", "Supplier parity rate", "Price Parity", "parity"),
#         ("tiqets_parity_rate", "Tiqets parity rate", "Price Parity", "parity"),
#         ("viator_parity_rate", "Viator parity rate", "Price Parity", "parity"),
#         ("headout_parity_rate", "Headout parity rate", "Price Parity", "parity"),
#         ("overall_parity_rate", "Overall parity rate", "Price Parity", "parity"),
#     ]
    
#     # Category descriptions
#     group_question = {
#         "Extensive Margin": (
#             "Extensive margin:\n"
#             "Did suppliers or tours\n"
#             "enter/exit the platform\n"
#             "after the change?"
#         ),
#         "Price Parity": (
#             "Price Parity:\n"
#             "Did price competitiveness\n"
#             "change after the event?"
#         ),
#     }
    
#     # Pivot data by period
#     supply_treat = supply_df[supply_df[period_col].str.contains(treat_period)].set_index(period_col)
#     supply_ctrl = supply_df[supply_df[period_col].str.contains(control_period)].set_index(period_col)
#     parity_treat = parity_df[parity_df[period_col].str.contains(treat_period)].set_index(period_col)
#     parity_ctrl = parity_df[parity_df[period_col].str.contains(control_period)].set_index(period_col)
    
#     # Calculate DID for each metric
#     eps = 1e-12
#     did_results = []
    
#     for metric_col, pretty_name, group, source in metrics_config:
#         # Select the right dataframe
#         if source == "supply":
#             treat = supply_treat
#             ctrl = supply_ctrl
#         else:
#             treat = parity_treat
#             ctrl = parity_ctrl
        
#         if metric_col not in treat.columns or metric_col not in ctrl.columns:
#             continue
            
#         pre_cy_key = f"{pre_window}_{treat_period}"
#         post_cy_key = f"{post_window}_{treat_period}"
#         pre_ly_key = f"{pre_window}_{control_period}"
#         post_ly_key = f"{post_window}_{control_period}"
        
#         if pre_cy_key in treat.index and post_cy_key in treat.index and \
#            pre_ly_key in ctrl.index and post_ly_key in ctrl.index:
            
#             treat_pct = (treat.loc[post_cy_key, metric_col] - treat.loc[pre_cy_key, metric_col]) / (treat.loc[pre_cy_key, metric_col] + eps)
#             ctrl_pct = (ctrl.loc[post_ly_key, metric_col] - ctrl.loc[pre_ly_key, metric_col]) / (ctrl.loc[pre_ly_key, metric_col] + eps)
#             did_pct = treat_pct - ctrl_pct
            
#             did_results.append({
#                 "metric": metric_col,
#                 "pretty": pretty_name,
#                 "group": group,
#                 "did_pct": did_pct * 100,  # Convert to percentage
#             })
    
#     if not did_results:
#         print("No DID results calculated. Check your data structure.")
#         return None, None
    
#     did_df = pd.DataFrame(did_results)
    
#     # Order by groups
#     group_order = ["Extensive Margin", "Price Parity"]
#     ordered_metrics = []
#     for g in group_order:
#         ordered_metrics.extend(did_df[did_df["group"] == g]["metric"].tolist())
    
#     y_map = {m: i for i, m in enumerate(ordered_metrics)}
#     y_labels = [did_df[did_df["metric"] == m]["pretty"].iloc[0] for m in ordered_metrics]
    
#     # Create plot
#     fig, ax = plt.subplots(1, 1, figsize=figsize)
    
#     xs = [did_df[did_df["metric"] == m]["did_pct"].iloc[0] for m in ordered_metrics]
#     ys = [y_map[m] for m in ordered_metrics]
    
#     ax.scatter(xs, ys, s=40, color="black", zorder=3)  # REDUCED size from 50 to 40
#     ax.axvline(0, color="red", linestyle="--", linewidth=1.5, zorder=2)
#     ax.grid(True, axis="x", color="0.9")
#     ax.set_title(f"{pre_window} vs {post_window}\n{country}", fontsize=12, fontweight="bold")  # REDUCED from 14
#     ax.set_xlabel(f"DID % change = (({treat_period} (POST−PRE)/PRE) − ({control_period} (POST−PRE)/PRE))", fontsize=9)  # REDUCED from 10
    
#     ax.set_yticks(range(len(ordered_metrics)))
#     ax.set_yticklabels(y_labels, fontsize=9)  # ADDED smaller font
#     ax.tick_params(axis='x', labelsize=9)  # ADDED smaller font for x-axis
#     ax.invert_yaxis()
    
#     # Group spans
#     group_spans = []
#     start = 0
#     for g in group_order:
#         ms = [m for m in ordered_metrics if did_df[did_df["metric"] == m]["group"].iloc[0] == g]
#         if not ms:
#             continue
#         end = start + len(ms) - 1
#         group_spans.append((g, start, end))
#         start = end + 1
    
#     # Separators between groups
#     for _, s, e in group_spans[:-1]:
#         ax.axhline(e + 0.5, color="0.5", linewidth=1.5)  # REDUCED from 2
    
#     # Right-side category strip
#     strip_ax = fig.add_axes([strip_left, 0.12, strip_width, 0.75])  # ADJUSTED positioning
#     strip_ax.set_xlim(0, 1)
#     strip_ax.set_ylim(-0.5, len(ordered_metrics) - 0.5)
#     strip_ax.invert_yaxis()
#     strip_ax.axis("off")
    
#     for g, s, e in group_spans:
#         rect = Rectangle((0, s - 0.5), 1, (e - s + 1), facecolor="0.92", edgecolor="0.3", linewidth=1)
#         strip_ax.add_patch(rect)
        
#         label = group_question.get(g, g)
#         strip_ax.text(
#             0.5, (s + e) / 2,
#             label,
#             ha="center", va="center",
#             fontsize=strip_fontsize,
#             linespacing=1.2,  # REDUCED from 1.3
#         )
    
#     plt.tight_layout(rect=[0.05, 0.05, strip_left - 0.01, 0.95])
    
#     return fig, did_df


# # Cell 7: Generate Chart
# fig, did_df = plot_combined_did_chart(supply_df, parity_df, country)

# if fig:
#     plt.show()
#     print("\nDID Results:")
#     print(did_df.to_string(index=False))
# else:
#     print("Chart generation failed. Check data structure.")

# # Cell 8: Summary Table
# if did_df is not None:
#     print("\n" + "="*80)
#     print(f"DIFFERENCE-IN-DIFFERENCES ANALYSIS: {country}")
#     print("="*80)
    
#     for _, row in did_df.iterrows():
#         print(f"\n{row['pretty']}: {row['did_pct']:.2f}%")
#         if row['did_pct'] > 5:
#             print("  → STRONG POSITIVE treatment effect")
#         elif row['did_pct'] > 0:
#             print("  → Positive treatment effect")
#         elif row['did_pct'] > -5:
#             print("  → Negative treatment effect")
#         else:
#             print("  → STRONG NEGATIVE treatment effect")
    
#     print("\n" + "="*80)


In [0]:
# Cell 1: Imports
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore", message="findfont")

# Cell 2: Create Widget
dbutils.widgets.text("country", "Germany", "Country")

# Cell 3: Get Country Parameter
country = dbutils.widgets.get("country")
print(f"Selected Country: {country}")

# Cell 4: Load Supply Data
supply_query = f"""
SELECT *
FROM production.supply_analytics.dst_supply_dataset
WHERE country_name = '{country}'
"""

supply_df = spark.sql(supply_query).toPandas()
print(f"Loaded {len(supply_df)} supply rows for {country}")

# Cell 5: Load Price Parity Data
parity_query = f"""
SELECT *
FROM production.supply_analytics.dst_price_parity
WHERE country_name = '{country}'
"""

parity_df = spark.sql(parity_query).toPandas()
print(f"Loaded {len(parity_df)} parity rows for {country}")

# Cell 6: Load Price Change Data
price_change_query = f"""
SELECT *
FROM production.supply_analytics.dst_price_change
WHERE country_name = '{country}'
"""

price_change_df = spark.sql(price_change_query).toPandas()
print(f"Loaded {len(price_change_df)} price change rows for {country}")

# Cell 7: Load Booking Data
booking_query = f"""
SELECT *
FROM production.supply_analytics.dst_booking_dataset
WHERE country_name = '{country}'
"""

booking_df = spark.sql(booking_query).toPandas()
print(f"Loaded {len(booking_df)} booking rows for {country}")

# Cell 8: Load Cancellation Data
cancellation_query = f"""
SELECT *
FROM production.supply_analytics.dst_cancellation_dataset
WHERE country_name = '{country}'
"""

cancellation_df = spark.sql(cancellation_query).toPandas()
print(f"Loaded {len(cancellation_df)} cancellation rows for {country}")

# Cell 9: Load Session Performance Data
session_query = f"""
SELECT *
FROM production.supply_analytics.dst_session_performance_dataset
WHERE country_name = '{country}'
"""

session_df = spark.sql(session_query).toPandas()
print(f"Loaded {len(session_df)} session rows for {country}")

# Convert all numeric columns to float
for df in [supply_df, parity_df, price_change_df, booking_df, cancellation_df, session_df]:
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        df[col] = df[col].astype(float)

print("\nSupply data:")
print(supply_df)
print("\nParity data:")
print(parity_df)
print("\nPrice change data:")
print(price_change_df)
print("\nBooking data:")
print(booking_df)
print("\nCancellation data:")
print(cancellation_df)
print("\nSession performance data:")
print(session_df)


# =============================================================================
# Cell 10: Calculate DID Results
# =============================================================================
def calculate_did_results(
    supply_df,
    parity_df,
    price_change_df,
    booking_df,
    cancellation_df,
    session_df,
    period_col="period_window",
    treat_period="CY",
    control_period="LY",
    pre_window="PRE",
    post_window="POST",
):
    """
    Calculate Difference-in-Differences estimates for all metrics.

    DID % = (CY POST - CY PRE) / CY PRE  -  (LY POST - LY PRE) / LY PRE
    """

    metrics_config = [
        # Extensive Margin
        ("total_tours",              "Total tours",                    "Extensive Margin", "supply"),
        ("active_tours",             "Active tours",                   "Extensive Margin", "supply"),
        ("share_active_tours",       "Share active tours",             "Extensive Margin", "supply"),
        ("total_suppliers",          "Total suppliers",                "Extensive Margin", "supply"),
        ("active_suppliers",         "Active suppliers",               "Extensive Margin", "supply"),
        ("share_active_suppliers",   "Share active suppliers",         "Extensive Margin", "supply"),
        ("avg_days_online_per_tour_per_week",     "Avg days online per tour/week",       "Extensive Margin", "supply"),
        ("avg_days_online_per_supplier_per_week", "Avg days online per supplier/week",   "Extensive Margin", "supply"),
        # Booking Performance
        ("bookings",                 "Bookings",                       "Booking Performance", "booking"),
        ("tickets",                  "Tickets sold",                   "Booking Performance", "booking"),
        ("nr",                       "Net revenue",                    "Booking Performance", "booking"),
        ("gmv",                      "GMV",                            "Booking Performance", "booking"),
        ("supplier_share_revenue",   "Supplier share revenue",         "Booking Performance", "booking"),
        ("gmv_supplier",             "GMV supplier",                   "Booking Performance", "booking"),
        ("avg_gmv_per_ticket",       "Avg GMV per ticket",             "Booking Performance", "booking"),
        ("avg_nr_per_ticket",        "Avg NR per ticket",              "Booking Performance", "booking"),
        ("avg_gmv_per_booking",      "Avg GMV per booking",            "Booking Performance", "booking"),
        ("avg_nr_per_booking",       "Avg NR per booking",             "Booking Performance", "booking"),
        # Cancellations
        ("cancellation_rate",        "Cancellation rate (all)",        "Cancellations", "cancellation"),
        ("cancellation_rate_nr",     "Cancellation rate NR (all)",     "Cancellations", "cancellation"),
        ("cancellation_rate_3m",     "Cancellation rate (3m travel)",  "Cancellations", "cancellation"),
        ("cancellation_rate_nr_3m",  "Cancellation rate NR (3m travel)","Cancellations", "cancellation"),
        ("cancellation_rate_6m",     "Cancellation rate (6m travel)",  "Cancellations", "cancellation"),
        ("cancellation_rate_nr_6m",  "Cancellation rate NR (6m travel)","Cancellations", "cancellation"),
        ("cancellation_rate_12m",    "Cancellation rate (12m travel)", "Cancellations", "cancellation"),
        ("cancellation_rate_nr_12m", "Cancellation rate NR (12m travel)","Cancellations", "cancellation"),
        # Customer Response (Session Performance)
        ("impressions",              "Impressions",                    "Customer Response", "session"),
        ("clicks",                   "Clicks",                         "Customer Response", "session"),
        ("visitors",                 "Visitors",                       "Customer Response", "session"),
        ("customers",                "Customers",                      "Customer Response", "session"),
        ("click_through_rate",       "Click-through rate",             "Customer Response", "session"),
        ("conversion_rate",          "Conversion rate",                "Customer Response", "session"),
        ("add_to_cart_rate",         "Add-to-cart rate",               "Customer Response", "session"),
        ("unavailability_issue_rate","Unavailability issue rate",      "Customer Response", "session"),
        ("avg_star_rating",          "Avg star rating",                "Customer Response", "session"),
        ("bookings_per_click",       "Bookings per click",             "Customer Response", "session"),
        ("avg_impressions_per_tour", "Avg impressions per tour",       "Customer Response", "session"),
        # Price Parity
        ("supplier_parity_rate",     "Supplier parity rate",           "Price Parity",     "parity"),
        ("tiqets_parity_rate",       "Tiqets parity rate",             "Price Parity",     "parity"),
        ("viator_parity_rate",       "Viator parity rate",             "Price Parity",     "parity"),
        ("overall_parity_rate",      "Overall parity rate",            "Price Parity",     "parity"),
        # Price Change
        ("pct_suppliers_changed_3m", "Suppliers changed price (3m)",   "Price Change",     "price_change"),
        ("pct_suppliers_changed_6m", "Suppliers changed price (6m)",   "Price Change",     "price_change"),
        ("pct_suppliers_changed_12m","Suppliers changed price (12m)",  "Price Change",     "price_change"),
        ("pct_tours_changed_3m",     "Tours with price change (3m)",   "Price Change",     "price_change"),
        ("pct_tours_changed_6m",     "Tours with price change (6m)",   "Price Change",     "price_change"),
        ("pct_tours_changed_12m",    "Tours with price change (12m)",  "Price Change",     "price_change"),
        ("median_final_red_3m",      "Median price 3m",                "Price Change",     "price_change"),
        ("median_final_red_6m",      "Median price 6m",                "Price Change",     "price_change"),
        ("median_final_red_12m",     "Median price 12m",               "Price Change",     "price_change"),
        # Availability
        ("median_timeslots_3m",      "Median timeslots 3m",            "Availability",     "price_change"),
        ("median_timeslots_6m",      "Median timeslots 6m",            "Availability",     "price_change"),
        ("median_timeslots_12m",     "Median timeslots 12m",           "Availability",     "price_change"),
    ]

    def get_period_col(df, default="period_window"):
        if default in df.columns:
            return default
        for col in df.columns:
            if 'period' in col.lower() or 'window' in col.lower():
                return col
        return default

    supply_pcol       = get_period_col(supply_df, period_col)
    parity_pcol       = get_period_col(parity_df, period_col)
    price_pcol        = get_period_col(price_change_df, period_col)
    booking_pcol      = get_period_col(booking_df, period_col)
    cancellation_pcol = get_period_col(cancellation_df, period_col)
    session_pcol      = get_period_col(session_df, period_col)

    supply_treat = supply_df[supply_df[supply_pcol].str.contains(treat_period)].set_index(supply_pcol)
    supply_ctrl  = supply_df[supply_df[supply_pcol].str.contains(control_period)].set_index(supply_pcol)

    parity_treat = parity_df[parity_df[parity_pcol].str.contains(treat_period)].set_index(parity_pcol) if len(parity_df) > 0 else pd.DataFrame()
    parity_ctrl  = parity_df[parity_df[parity_pcol].str.contains(control_period)].set_index(parity_pcol) if len(parity_df) > 0 else pd.DataFrame()

    price_treat = price_change_df[price_change_df[price_pcol].str.contains(treat_period)].set_index(price_pcol)
    price_ctrl  = price_change_df[price_change_df[price_pcol].str.contains(control_period)].set_index(price_pcol)

    booking_treat = booking_df[booking_df[booking_pcol].str.contains(treat_period)].set_index(booking_pcol)
    booking_ctrl  = booking_df[booking_df[booking_pcol].str.contains(control_period)].set_index(booking_pcol)

    cancellation_treat = cancellation_df[cancellation_df[cancellation_pcol].str.contains(treat_period)].set_index(cancellation_pcol)
    cancellation_ctrl  = cancellation_df[cancellation_df[cancellation_pcol].str.contains(control_period)].set_index(cancellation_pcol)

    session_treat = session_df[session_df[session_pcol].str.contains(treat_period)].set_index(session_pcol)
    session_ctrl  = session_df[session_df[session_pcol].str.contains(control_period)].set_index(session_pcol)

    eps = 1e-12
    did_results = []

    for metric_col, pretty_name, group, source in metrics_config:
        if source == "supply":
            treat, ctrl = supply_treat, supply_ctrl
        elif source == "parity":
            treat, ctrl = parity_treat, parity_ctrl
        elif source == "booking":
            treat, ctrl = booking_treat, booking_ctrl
        elif source == "cancellation":
            treat, ctrl = cancellation_treat, cancellation_ctrl
        elif source == "session":
            treat, ctrl = session_treat, session_ctrl
        else:
            treat, ctrl = price_treat, price_ctrl

        if len(treat) == 0 or len(ctrl) == 0:
            print(f"  SKIP {pretty_name}: {source} data is empty")
            continue

        if metric_col not in treat.columns or metric_col not in ctrl.columns:
            print(f"  SKIP {pretty_name}: column '{metric_col}' not found in {source}")
            continue

        pre_cy  = f"{pre_window}_{treat_period}"
        post_cy = f"{post_window}_{treat_period}"
        pre_ly  = f"{pre_window}_{control_period}"
        post_ly = f"{post_window}_{control_period}"

        if pre_cy not in treat.index or post_cy not in treat.index or \
           pre_ly not in ctrl.index or post_ly not in ctrl.index:
            print(f"  SKIP {pretty_name}: missing period keys")
            continue

        raw_pre_ly  = ctrl.loc[pre_ly, metric_col]
        raw_post_ly = ctrl.loc[post_ly, metric_col]
        raw_pre_cy  = treat.loc[pre_cy, metric_col]
        raw_post_cy = treat.loc[post_cy, metric_col]

        if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in [raw_pre_ly, raw_post_ly, raw_pre_cy, raw_post_cy]):
            print(f"  SKIP {pretty_name}: one or more period values are NULL/NaN")
            continue

        pre_ly_val  = float(raw_pre_ly)
        post_ly_val = float(raw_post_ly)
        pre_cy_val  = float(raw_pre_cy)
        post_cy_val = float(raw_post_cy)

        treat_pct = (post_cy_val - pre_cy_val) / (pre_cy_val + eps)
        ctrl_pct  = (post_ly_val - pre_ly_val) / (pre_ly_val + eps)
        did_pct   = treat_pct - ctrl_pct

        did_results.append({
            "metric": metric_col,
            "pretty": pretty_name,
            "group": group,
            "did_pct": did_pct * 100,
            "pre_ly": pre_ly_val,
            "post_ly": post_ly_val,
            "pre_cy": pre_cy_val,
            "post_cy": post_cy_val,
        })
        print(f"  OK {pretty_name} [{group}]: DID = {did_pct * 100:+.1f}pp")

    return pd.DataFrame(did_results) if did_results else None


# =============================================================================
# Cell 11: DID Visualization (Management-Ready)
# =============================================================================
def plot_did_analysis(did_df, country):
    """
    Single management-ready DID horizontal bar chart.
    """

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.size"] = 11

    # ---- Section configuration ----
    group_order = [
        "Extensive Margin",
        "Booking Performance",
        "Cancellations",
        "Customer Response",
        "Price Parity",
        "Price Change",
        "Availability",
    ]

    group_config = {
        "Extensive Margin": {
            "title": "EXTENSIVE MARGIN",
            "subtitle": "Did suppliers or tours enter\nor exit the platform?",
            "bg": "#E8F5E9",
            "border": "#66BB6A",
        },
        "Booking Performance": {
            "title": "BOOKING PERFORMANCE",
            "subtitle": "Did booking volume, revenue,\nor prices change after the event?",
            "bg": "#FFFDE7",
            "border": "#FFD54F",
        },
        "Cancellations": {
            "title": "CANCELLATIONS",
            "subtitle": "Did cancellation rates change\nfor future travel dates?",
            "bg": "#FFEBEE",
            "border": "#EF5350",
        },
        "Customer Response": {
            "title": "CUSTOMER RESPONSE",
            "subtitle": "Did visitor engagement,\nconversion, or ratings change?",
            "bg": "#E8EAF6",
            "border": "#5C6BC0",
        },
        "Price Parity": {
            "title": "PRICE PARITY",
            "subtitle": "Did price competitiveness\nchange after the event?",
            "bg": "#E3F2FD",
            "border": "#42A5F5",
        },
        "Price Change": {
            "title": "PRICE CHANGE",
            "subtitle": "Did suppliers adjust their\nprices after the event?",
            "bg": "#FFF3E0",
            "border": "#FFA726",
        },
        "Availability": {
            "title": "AVAILABILITY",
            "subtitle": "Did the number of available\ntimeslots change after the event?",
            "bg": "#F3E5F5",
            "border": "#AB47BC",
        },
    }

    threshold = 2  # percentage points

    def bar_color(val):
        if val > threshold:
            return "#2E7D32"  # dark green
        elif val < -threshold:
            return "#C62828"  # dark red
        else:
            return "#757575"  # gray

    # ---- Build ordered metric list ----
    active_groups = [g for g in group_order if len(did_df[did_df["group"] == g]) > 0]

    all_pretty = []
    all_did = []
    section_ranges = []

    pos = 0
    for g in active_groups:
        gdf = did_df[did_df["group"] == g]
        start = pos
        for _, row in gdf.iterrows():
            all_pretty.append(row["pretty"])
            all_did.append(row["did_pct"])
            pos += 1
        section_ranges.append((g, start, pos))

    n = len(all_pretty)

    # Reverse so first group appears at the top
    all_pretty_r = all_pretty[::-1]
    all_did_r = all_did[::-1]
    y = np.arange(n)

    # Reversed section ranges for drawing
    rev_ranges = [(g, n - end, n - start) for g, start, end in section_ranges]

    colors = [bar_color(d) for d in all_did_r]

    # ---- Figure: main chart + annotation column ----
    fig = plt.figure(figsize=(16, max(10, n * 0.6)))
    gs = GridSpec(1, 2, width_ratios=[5, 1], wspace=0.02)

    ax = fig.add_subplot(gs[0])
    ax_annot = fig.add_subplot(gs[1], sharey=ax)
    ax_annot.axis("off")

    # ---- Background shading per section ----
    for g, s, e in rev_ranges:
        cfg = group_config[g]
        ax.axhspan(s - 0.5, e - 0.5, facecolor=cfg["bg"], alpha=0.35, zorder=0)
        ax_annot.axhspan(s - 0.5, e - 0.5, facecolor=cfg["bg"], alpha=0.35, zorder=0)
        if s > 0:
            ax.axhline(y=s - 0.5, color="#BDBDBD", linewidth=1.0, zorder=1)
            ax_annot.axhline(y=s - 0.5, color="#BDBDBD", linewidth=1.0, zorder=1)

    # ---- Horizontal bars ----
    ax.barh(y, all_did_r, height=0.55, color=colors, edgecolor="white",
            linewidth=0.3, alpha=0.9, zorder=2)

    # ---- Value labels ----
    for i, (val, yp) in enumerate(zip(all_did_r, y)):
        sign = "+" if val > 0 else ""
        label = f"{sign}{val:.0f}pp"
        if val >= 0:
            ax.text(val + 0.8, yp, label, va="center", ha="left",
                    fontsize=10, fontweight="bold", color=colors[i])
        else:
            ax.text(val - 0.8, yp, label, va="center", ha="right",
                    fontsize=10, fontweight="bold", color=colors[i])

    # ---- Zero line ----
    ax.axvline(x=0, color="#333333", linewidth=1.2, zorder=3)

    # ---- Section labels in annotation column ----
    for g, s, e in rev_ranges:
        cfg = group_config[g]
        mid_y = (s + e - 1) / 2
        ax_annot.text(
            0.5, mid_y,
            f"{cfg['title']}\n{cfg['subtitle']}",
            va="center", ha="center", fontsize=9, color="#424242",
            fontstyle="italic",
            bbox=dict(
                boxstyle="round,pad=0.6", facecolor=cfg["bg"],
                edgecolor=cfg["border"], alpha=0.95, linewidth=1.5,
            ),
            transform=ax_annot.transData,
        )

    # ---- Axes formatting ----
    ax.set_yticks(y)
    ax.set_yticklabels(all_pretty_r, fontsize=10.5)

    max_abs = max(abs(d) for d in all_did_r) * 1.3
    ax.set_xlim(-max_abs, max_abs)

    ax.set_xlabel(
        "DID Estimate (percentage points)\n"
        "Positive values mean the current year improved more than last year  |  "
        "Negative values mean a relative decline compared to last year",
        fontsize=9, color="#555555", labelpad=12,
    )

    # ---- Title ----
    fig.suptitle(
        f"{country}: Difference-in-Differences Analysis",
        fontsize=16, fontweight="bold", color="#1a1a1a",
        x=0.08, ha="left", y=0.97,
    )
    ax.set_title(
        "Comparing PRE to POST changes between Current Year and Last Year\n",
        fontsize=11, color="#555555", loc="left", pad=8,
    )

    # ---- Legend ----
    legend_elements = [
        mpatches.Patch(facecolor="#2E7D32", label=f"Positive shift (> +{threshold}pp)"),
        mpatches.Patch(facecolor="#757575", label=f"No meaningful change (-{threshold} to +{threshold}pp)"),
        mpatches.Patch(facecolor="#C62828", label=f"Negative shift (< -{threshold}pp)"),
    ]
    ax.legend(
        handles=legend_elements, loc="lower left", fontsize=9,
        framealpha=0.95, edgecolor="#BDBDBD", fancybox=True, borderpad=1,
    )

    # ---- How-to-read guide ----
    fig.text(
        0.08, 0.015,
        "How to read: Each bar shows the difference in PRE-to-POST percentage change between this year and last year. "
        "Example: Active tours = +8pp means the growth from PRE to POST was 8 percentage points larger this year than last year.",
        fontsize=8, color="#888888", style="italic", ha="left", va="bottom",
    )

    # ---- Clean up spines ----
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)
    ax.tick_params(axis="both", which="both", length=0)
    ax.grid(axis="x", alpha=0.15, linestyle="--", zorder=0)

    plt.tight_layout(rect=[0, 0.04, 1, 0.95])

    return fig


# =============================================================================
# Cell 12: Execute
# =============================================================================
did_df = calculate_did_results(supply_df, parity_df, price_change_df, booking_df, cancellation_df, session_df)

if did_df is None:
    print("\nERROR: No DID results could be calculated. Check data above.")
else:
    print(f"\n{'='*60}")
    print("RESULTS SUMMARY:")
    for g in ["Extensive Margin", "Booking Performance", "Cancellations", "Customer Response", "Price Parity", "Price Change", "Availability"]:
        n = len(did_df[did_df["group"] == g])
        print(f"  {g}: {n} metrics")
    print(f"{'='*60}")
    print("\nDID Results Table:")
    print(did_df[["pretty", "group", "did_pct"]].to_string(index=False))

    fig = plot_did_analysis(did_df, country)
    plt.show()
